# EDA complet de `blunders.csv`


In [1]:
from pathlib import Path

INPUT_CSV = "output/blunders.csv"
OUTPUT_DIR = Path("output/blunders_eda")
TOP_N = 15
MIN_GROUP_SIZE = 25

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR = OUTPUT_DIR / "tables"
PLOTS_DIR = OUTPUT_DIR / "plots"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
import math
from pathlib import Path
from typing import Dict, List, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NUMERIC_COLUMNS = [
    "target_elo",
    "opponent_elo",
    "elo_diff",
    "cp_loss",
    "eval_player_before_cp",
    "eval_player_after_cp",
    "material_balance_before_cp",
    "material_balance_after_move_cp",
    "material_balance_delta_cp",
    "hanging_pieces_count_after",
    "king_danger_before",
    "king_danger_after",
    "king_danger_delta",
    "king_escape_squares_after",
    "checks_available_to_opponent_after",
]

BOOLEAN_COLUMNS = [
    "is_capture",
    "is_check",
    "is_castle",
    "is_promotion",
    "piece_hung_after_move",
    "lost_castling_rights",
    "back_rank_weak_after",
    "opponent_has_en_passant_after",
    "opponent_has_double_attack_after",
    "opponent_has_fork_after",
    "opponent_has_knight_fork_after",
    "opponent_has_pawn_fork_after",
    "opponent_has_discovered_attack_after",
    "opponent_can_trap_piece_after_move",
    "opponent_has_smothered_mate_pattern_after",
]

TACTICAL_FLAGS = [
    "piece_hung_after_move",
    "lost_castling_rights",
    "back_rank_weak_after",
    "opponent_has_en_passant_after",
    "opponent_has_double_attack_after",
    "opponent_has_fork_after",
    "opponent_has_knight_fork_after",
    "opponent_has_pawn_fork_after",
    "opponent_has_discovered_attack_after",
    "opponent_can_trap_piece_after_move",
    "opponent_has_smothered_mate_pattern_after",
]

TEXT_COLUMNS = [
    "phase",
    "speed",
    "opening_name",
    "target_result",
    "move_quality",
    "promotion_piece",
    "hung_piece_type",
    "eval_state_before",
    "eval_state_after",
    "eval_state_transition",
    "move_san",
    "move_uci",
]

## Fonctions utilitaires

In [3]:
def safe_rate(series: pd.Series) -> float:
    if len(series) == 0:
        return 0.0
    return float(series.mean())


def percent(x: float) -> str:
    return f"{100.0 * float(x):.1f}%"


def save_table(df_or_series, path: Path) -> None:
    if isinstance(df_or_series, pd.Series):
        df_or_series.to_csv(path, header=True)
    else:
        df_or_series.to_csv(path, index=False)


def top_categories(df: pd.DataFrame, column: str, top_n: int = TOP_N) -> pd.DataFrame:
    if column not in df.columns:
        return pd.DataFrame(columns=[column, "count", "share"])
    counts = (
        df[column]
        .fillna("")
        .astype(str)
        .replace("", "<missing>")
        .value_counts(dropna=False)
        .head(top_n)
        .rename_axis(column)
        .reset_index(name="count")
    )
    counts["share"] = (counts["count"] / len(df)).round(4) if len(df) else 0.0
    return counts


def describe_numeric(df: pd.DataFrame, columns: Sequence[str]) -> pd.DataFrame:
    rows = []
    for col in columns:
        if col not in df.columns:
            continue
        s = pd.to_numeric(df[col], errors="coerce").dropna()
        if s.empty:
            continue
        rows.append(
            {
                "feature": col,
                "count": int(s.shape[0]),
                "mean": round(float(s.mean()), 3),
                "median": round(float(s.median()), 3),
                "std": round(float(s.std(ddof=1)) if s.shape[0] > 1 else 0.0, 3),
                "min": round(float(s.min()), 3),
                "p10": round(float(s.quantile(0.10)), 3),
                "p90": round(float(s.quantile(0.90)), 3),
                "max": round(float(s.max()), 3),
            }
        )
    return pd.DataFrame(rows)


def summarize_missingness(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    n = len(df)
    for col in df.columns:
        missing = int(df[col].isna().sum())
        empty = 0
        if df[col].dtype == object:
            empty = int(df[col].astype(str).str.strip().eq("").sum())
        rows.append(
            {
                "column": col,
                "missing": missing,
                "missing_rate": round(missing / n, 4) if n else 0.0,
                "empty_strings": empty,
            }
        )
    return pd.DataFrame(rows).sort_values(["missing_rate", "empty_strings"], ascending=False).reset_index(drop=True)


def summarize_boolean_flags(df: pd.DataFrame, cols: Sequence[str]) -> pd.DataFrame:
    rows = []
    for col in cols:
        if col not in df.columns:
            continue
        rows.append(
            {
                "feature": col,
                "count_true": int(df[col].sum()),
                "rate_true": round(float(df[col].mean()), 4) if len(df) else 0.0,
            }
        )
    return pd.DataFrame(rows).sort_values(["rate_true", "count_true"], ascending=False).reset_index(drop=True)


def bool_rate_by_group(df: pd.DataFrame, group_col: str, flag_col: str) -> pd.DataFrame:
    if group_col not in df.columns or flag_col not in df.columns:
        return pd.DataFrame()
    out = (
        df.groupby(group_col, dropna=False)[flag_col]
        .agg(count="size", rate="mean")
        .reset_index()
        .sort_values(["rate", "count"], ascending=[False, False])
    )
    out["rate"] = out["rate"].round(4)
    return out


def correlation_table(df: pd.DataFrame) -> pd.DataFrame:
    numeric = [c for c in NUMERIC_COLUMNS if c in df.columns]
    if "cp_loss" not in numeric:
        return pd.DataFrame(columns=["feature", "corr_with_cp_loss"])
    rows = []
    for col in numeric:
        if col == "cp_loss":
            continue
        subset = df[["cp_loss", col]].dropna()
        if subset.shape[0] < 5:
            continue
        corr = subset["cp_loss"].corr(subset[col])
        rows.append({"feature": col, "corr_with_cp_loss": round(float(corr), 4) if pd.notna(corr) else np.nan})
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    return out.reindex(out["corr_with_cp_loss"].abs().sort_values(ascending=False).index).reset_index(drop=True)


def top_hung_pieces(df: pd.DataFrame, top_n: int = TOP_N) -> pd.DataFrame:
    if "hung_piece_type" not in df.columns:
        return pd.DataFrame()
    subset = df.loc[df["piece_hung_after_move"] == True].copy() if "piece_hung_after_move" in df.columns else df.copy()
    if subset.empty:
        return pd.DataFrame(columns=["hung_piece_type", "count", "share_within_hangs", "mean_cp_loss"])
    out = (
        subset["hung_piece_type"]
        .replace("", np.nan)
        .dropna()
        .value_counts()
        .head(top_n)
        .rename_axis("hung_piece_type")
        .reset_index(name="count")
    )
    out["share_within_hangs"] = (out["count"] / len(subset)).round(4)
    if "cp_loss" in subset.columns:
        means = subset.groupby("hung_piece_type")["cp_loss"].mean().round(3).rename("mean_cp_loss")
        out = out.merge(means, on="hung_piece_type", how="left")
    return out.sort_values(["count"], ascending=[False]).reset_index(drop=True)


def group_quality_stats(df: pd.DataFrame) -> pd.DataFrame:
    if "move_quality" not in df.columns:
        return pd.DataFrame()
    cols = [c for c in ["cp_loss", "king_danger_delta", "checks_available_to_opponent_after", "hanging_pieces_count_after"] if c in df.columns]
    grouped = df.groupby("move_quality", dropna=False)
    rows = []
    for key, g in grouped:
        row = {"move_quality": key, "count": int(len(g))}
        for col in cols:
            row[f"{col}_mean"] = round(float(g[col].mean()), 3) if g[col].notna().any() else np.nan
            row[f"{col}_median"] = round(float(g[col].median()), 3) if g[col].notna().any() else np.nan
        for flag in TACTICAL_FLAGS:
            if flag in g.columns:
                row[f"{flag}_rate"] = round(safe_rate(g[flag]), 4)
        rows.append(row)
    return pd.DataFrame(rows).sort_values("count", ascending=False).reset_index(drop=True)


def slice_ranking(df: pd.DataFrame, group_cols: Sequence[str], min_group_size: int = MIN_GROUP_SIZE) -> pd.DataFrame:
    usable = [c for c in group_cols if c in df.columns]
    if not usable or "cp_loss" not in df.columns:
        return pd.DataFrame()

    agg_map = {
        "count": ("cp_loss", "size"),
        "mean_cp_loss": ("cp_loss", "mean"),
        "median_cp_loss": ("cp_loss", "median"),
    }
    if "king_danger_delta" in df.columns:
        agg_map["mean_king_danger_delta"] = ("king_danger_delta", "mean")

    grouped = df.groupby(list(usable), dropna=False).agg(**agg_map).reset_index()
    grouped = grouped[grouped["count"] >= min_group_size].copy()

    if "piece_hung_after_move" in df.columns:
        hang_rate = df.groupby(list(usable), dropna=False)["piece_hung_after_move"].mean().rename("piece_hung_rate")
        grouped = grouped.merge(hang_rate.reset_index(), on=list(usable), how="left")
    if "opponent_has_fork_after" in df.columns:
        fork_rate = df.groupby(list(usable), dropna=False)["opponent_has_fork_after"].mean().rename("fork_rate")
        grouped = grouped.merge(fork_rate.reset_index(), on=list(usable), how="left")

    for col in grouped.columns:
        if col.startswith("mean_") or col.endswith("_rate"):
            grouped[col] = grouped[col].round(4)

    return grouped.sort_values(["mean_cp_loss", "count"], ascending=[False, False]).reset_index(drop=True)


def examples_table(df: pd.DataFrame, n: int = 25) -> pd.DataFrame:
    keep = [
        c for c in [
            "phase",
            "speed",
            "opening_name",
            "target_elo",
            "opponent_elo",
            "move_san",
            "cp_loss",
            "move_quality",
            "eval_state_transition",
            "piece_hung_after_move",
            "hung_piece_type",
            "king_danger_delta",
            "checks_available_to_opponent_after",
            "opponent_has_fork_after",
            "opponent_has_discovered_attack_after",
            "opponent_can_trap_piece_after_move",
            "opponent_has_smothered_mate_pattern_after",
        ] if c in df.columns
    ]
    if not keep or "cp_loss" not in df.columns:
        return pd.DataFrame()
    return df.sort_values("cp_loss", ascending=False).head(n)[keep].reset_index(drop=True)


def plot_bar(table: pd.DataFrame, x: str, y: str, path: Path, title: str, top_n: int | None = None, rotate: bool = False) -> None:
    if table.empty or x not in table.columns or y not in table.columns:
        return
    data = table.head(top_n) if top_n else table
    plt.figure(figsize=(10, 5))
    plt.bar(data[x].astype(str), data[y])
    plt.title(title)
    plt.xlabel(x)
    plt.ylabel(y)
    if rotate:
        plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.show()
    plt.close()


def plot_hist(series: pd.Series, path: Path, title: str, bins: int = 30) -> None:
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return
    plt.figure(figsize=(8, 5))
    plt.hist(s, bins=bins)
    plt.title(title)
    plt.xlabel(series.name or "value")
    plt.ylabel("count")
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.show()
    plt.close()


def plot_box_by_group(df: pd.DataFrame, value_col: str, group_col: str, path: Path, title: str) -> None:
    if value_col not in df.columns or group_col not in df.columns:
        return
    groups = []
    labels = []
    for key, g in df.groupby(group_col, dropna=False):
        s = pd.to_numeric(g[value_col], errors="coerce").dropna()
        if s.empty:
            continue
        groups.append(s.values)
        labels.append(str(key))
    if not groups:
        return
    plt.figure(figsize=(10, 5))
    plt.boxplot(groups, labels=labels, showfliers=False)
    plt.title(title)
    plt.xlabel(group_col)
    plt.ylabel(value_col)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.show()
    plt.close()


def plot_scatter(df: pd.DataFrame, x: str, y: str, path: Path, title: str) -> None:
    if x not in df.columns or y not in df.columns:
        return
    subset = df[[x, y]].dropna()
    if subset.empty:
        return
    plt.figure(figsize=(7, 5))
    plt.scatter(subset[x], subset[y], alpha=0.25, s=12)
    plt.title(title)
    plt.xlabel(x)
    plt.ylabel(y)
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.show()
    plt.close()

## Chargement et préparation du dataset

In [4]:
df = pd.read_csv(INPUT_CSV)

for col in NUMERIC_COLUMNS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

for col in BOOLEAN_COLUMNS:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.lower()
            .isin({"1", "true", "yes", "y", "t"})
        )

for col in TEXT_COLUMNS:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str).str.strip()

if "target_elo" in df.columns:
    df["target_elo_bucket"] = pd.cut(
        df["target_elo"],
        bins=[0, 1200, 1400, 1600, 1800, 2000, 2200, 3000],
        right=False,
        include_lowest=True,
    ).astype(str)

if "cp_loss" in df.columns:
    df["cp_loss_bucket"] = pd.cut(
        df["cp_loss"],
        bins=[0, 100, 200, 300, 500, 1000, np.inf],
        right=False,
        include_lowest=True,
    ).astype(str)

df.shape

(4281, 50)

In [5]:
df.head()

,phase,speed,rated,opening_eco,opening_name,opening_ply,opening_family,target_elo,opponent_elo,elo_diff,...,opponent_has_en_passant_after,opponent_has_double_attack_after,opponent_has_fork_after,opponent_has_knight_fork_after,opponent_has_pawn_fork_after,opponent_has_discovered_attack_after,opponent_can_trap_piece_after_move,opponent_has_smothered_mate_pattern_after,target_elo_bucket,cp_loss_bucket
0,endgame,blitz,True,D00,Queen's Pawn Game: Accelerated London System,3,D,1522,984,538,...,False,True,False,False,False,False,False,False,"[1400, 1600)","[1000.0, inf)"
1,endgame,bullet,True,B23,Sicilian Defense: Closed,4,B,1216,1299,-83,...,False,False,False,False,False,False,False,False,"[1200, 1400)","[1000.0, inf)"
2,middlegame,blitz,True,C16,"French Defense: Winawer Variation, Advance Var...",7,C,1589,1522,67,...,False,True,False,False,False,True,False,False,"[1400, 1600)","[1000.0, inf)"
3,middlegame,blitz,True,C16,"French Defense: Winawer Variation, Advance Var...",7,C,1589,1522,67,...,False,False,False,False,False,False,False,False,"[1400, 1600)","[1000.0, inf)"
4,middlegame,blitz,True,C16,"French Defense: Winawer Variation, Advance Var...",7,C,1589,1522,67,...,False,False,False,False,False,False,False,False,"[1400, 1600)","[1000.0, inf)"


## Qualité des données

In [6]:
missingness = summarize_missingness(df)
missingness.head(20)

,column,missing,missing_rate,empty_strings
0,promotion_piece,0,0.0,4202
1,hung_piece_type,0,0.0,2827
2,phase,0,0.0,0
3,speed,0,0.0,0
4,rated,0,0.0,0
5,opening_eco,0,0.0,0
6,opening_name,0,0.0,0
7,opening_ply,0,0.0,0
8,opening_family,0,0.0,0
9,target_elo,0,0.0,0


## Résumé numérique global

In [7]:
numeric_summary = describe_numeric(df, NUMERIC_COLUMNS)
numeric_summary

,feature,count,mean,median,std,min,p10,p90,max
0,target_elo,4281,1742.856,1789.00,316.801,747.0,1284.0,2117.00,2836.00
1,opponent_elo,4281,1755.470,1777.00,370.321,513.0,1246.0,2221.00,2879.00
2,elo_diff,4281,-12.613,-2.00,278.684,-1291.0,-364.0,341.00,1435.00
3,cp_loss,4281,3395.156,455.00,3878.129,100.0,114.0,9018.00,9595.00
4,eval_player_before_cp,4281,1842.185,631.00,4495.928,-8330.0,-1261.0,9988.00,9998.00
5,eval_player_after_cp,4281,-1552.971,471.00,4256.527,-9998.0,-9986.0,1277.00,8339.00
6,material_balance_before_cp,4281,1.588,0.00,684.628,-2430.0,-900.0,900.00,2470.00
7,material_balance_after_move_cp,4281,96.576,100.00,707.919,-2250.0,-800.0,1010.00,2470.00
8,material_balance_delta_cp,4281,94.987,0.00,210.727,0.0,0.0,500.00,1700.00
9,hanging_pieces_count_after,4281,0.422,0.00,0.656,0.0,0.0,1.00,4.00


## Répartition globale des erreurs

In [8]:
top_categories(df, "move_quality", TOP_N)

,move_quality,count,share
0,Blunder,2814,0.6573
1,Mistake,1467,0.3427


In [ ]:
plot_hist(df["cp_loss"], PLOTS_DIR / "cp_loss_hist.png", "Distribution de cp_loss")

## Analyse par phase, cadence, résultat, Elo

In [ ]:
phase_stats = pd.DataFrame()
if "phase" in df.columns:
    phase_stats = (
        df.groupby("phase", dropna=False)
        .agg(
            count=("cp_loss", "size"),
            mean_cp_loss=("cp_loss", "mean"),
            median_cp_loss=("cp_loss", "median"),
            hang_rate=("piece_hung_after_move", "mean") if "piece_hung_after_move" in df.columns else ("cp_loss", "size"),
            fork_rate=("opponent_has_fork_after", "mean") if "opponent_has_fork_after" in df.columns else ("cp_loss", "size"),
        )
        .reset_index()
        .sort_values(["mean_cp_loss", "count"], ascending=[False, False])
    )
    for col in ["mean_cp_loss", "median_cp_loss", "hang_rate", "fork_rate"]:
        if col in phase_stats.columns:
            phase_stats[col] = phase_stats[col].round(4)

phase_stats

In [ ]:
speed_stats = pd.DataFrame()
if "speed" in df.columns:
    speed_stats = (
        df.groupby("speed", dropna=False)
        .agg(
            count=("cp_loss", "size"),
            mean_cp_loss=("cp_loss", "mean"),
            median_cp_loss=("cp_loss", "median"),
        )
        .reset_index()
        .sort_values(["mean_cp_loss", "count"], ascending=[False, False])
    )
    for col in ["mean_cp_loss", "median_cp_loss"]:
        if col in speed_stats.columns:
            speed_stats[col] = speed_stats[col].round(4)

speed_stats

In [ ]:
result_stats = pd.DataFrame()
if "target_result" in df.columns:
    result_stats = (
        df.groupby("target_result", dropna=False)
        .agg(
            count=("cp_loss", "size"),
            mean_cp_loss=("cp_loss", "mean"),
            hang_rate=("piece_hung_after_move", "mean") if "piece_hung_after_move" in df.columns else ("cp_loss", "size"),
            fork_rate=("opponent_has_fork_after", "mean") if "opponent_has_fork_after" in df.columns else ("cp_loss", "size"),
        )
        .reset_index()
        .sort_values(["mean_cp_loss", "count"], ascending=[False, False])
    )
    for col in ["mean_cp_loss", "hang_rate", "fork_rate"]:
        if col in result_stats.columns:
            result_stats[col] = result_stats[col].round(4)

result_stats

In [ ]:
elo_bucket_stats = pd.DataFrame()
if "target_elo_bucket" in df.columns:
    elo_bucket_stats = (
        df.groupby("target_elo_bucket", dropna=False)
        .agg(
            count=("cp_loss", "size"),
            mean_cp_loss=("cp_loss", "mean"),
            mean_king_danger_delta=("king_danger_delta", "mean") if "king_danger_delta" in df.columns else ("cp_loss", "size"),
            hang_rate=("piece_hung_after_move", "mean") if "piece_hung_after_move" in df.columns else ("cp_loss", "size"),
            fork_rate=("opponent_has_fork_after", "mean") if "opponent_has_fork_after" in df.columns else ("cp_loss", "size"),
        )
        .reset_index()
        .sort_values("target_elo_bucket")
    )
    for col in ["mean_cp_loss", "mean_king_danger_delta", "hang_rate", "fork_rate"]:
        if col in elo_bucket_stats.columns:
            elo_bucket_stats[col] = elo_bucket_stats[col].round(4)

elo_bucket_stats

In [ ]:
if not phase_stats.empty:
    plot_bar(phase_stats, "phase", "mean_cp_loss", PLOTS_DIR / "phase_mean_cp_loss.png", "cp_loss moyen par phase")

if not speed_stats.empty:
    plot_bar(speed_stats, "speed", "mean_cp_loss", PLOTS_DIR / "speed_mean_cp_loss.png", "cp_loss moyen par cadence")

if not result_stats.empty:
    plot_bar(result_stats, "target_result", "mean_cp_loss", PLOTS_DIR / "target_result_mean_cp_loss.png", "cp_loss moyen selon le résultat final")

if not elo_bucket_stats.empty:
    plot_bar(elo_bucket_stats, "target_elo_bucket", "mean_cp_loss", PLOTS_DIR / "elo_bucket_mean_cp_loss.png", "cp_loss moyen par tranche Elo", rotate=True)

## Ouvertures, transitions d'évaluation et exemples extrêmes

In [ ]:
opening_name_stats = top_categories(df, "opening_name", TOP_N)
transition_stats = top_categories(df, "eval_state_transition", TOP_N)
opening_name_stats, transition_stats.head(10)

In [ ]:
worst_examples = examples_table(df, 25)
worst_examples.head(15)

## Pièces pendues et mécanismes matériels

In [ ]:
hung_piece_stats = top_hung_pieces(df, TOP_N)
hung_piece_stats

In [ ]:
if "hanging_pieces_count_after" in df.columns:
    plot_hist(
        df["hanging_pieces_count_after"],
        PLOTS_DIR / "hanging_pieces_count_hist.png",
        "Nombre de pièces pendues après le coup",
        bins=15,
    )

if not hung_piece_stats.empty:
    plot_bar(
        hung_piece_stats,
        "hung_piece_type",
        "count",
        PLOTS_DIR / "hung_piece_type_counts.png",
        "Pièces le plus souvent pendues",
    )

## Signaux de sécurité du roi

In [ ]:
king_table = describe_numeric(df, ["king_danger_before", "king_danger_after", "king_danger_delta", "king_escape_squares_after", "checks_available_to_opponent_after"])
king_table

In [ ]:
if "king_danger_delta" in df.columns:
    plot_hist(df["king_danger_delta"], PLOTS_DIR / "king_danger_delta_hist.png", "Distribution de king_danger_delta")

if "king_danger_delta" in df.columns and "cp_loss" in df.columns:
    plot_scatter(df, "king_danger_delta", "cp_loss", PLOTS_DIR / "king_danger_vs_cp_loss.png", "king_danger_delta vs cp_loss")

if "checks_available_to_opponent_after" in df.columns and "cp_loss" in df.columns:
    plot_scatter(df, "checks_available_to_opponent_after", "cp_loss", PLOTS_DIR / "checks_available_vs_cp_loss.png", "Checks disponibles vs cp_loss")

## Motifs tactiques booléens

In [ ]:
flag_summary = summarize_boolean_flags(df, TACTICAL_FLAGS)
flag_summary

In [ ]:
plot_bar(flag_summary, "feature", "rate_true", PLOTS_DIR / "tactical_flag_rates.png", "Taux des motifs tactiques", rotate=True)

In [ ]:
flag_tables = {}
for flag in TACTICAL_FLAGS:
    if flag in df.columns:
        flag_tables[f"{flag}_by_move_quality"] = bool_rate_by_group(df, "move_quality", flag)
        flag_tables[f"{flag}_by_phase"] = bool_rate_by_group(df, "phase", flag)
        flag_tables[f"{flag}_by_elo_bucket"] = bool_rate_by_group(df, "target_elo_bucket", flag)

next(iter(flag_tables.items())) if flag_tables else "Aucun flag exploitable" 

## Corrélations et pires sous-groupes

In [ ]:
correlations = correlation_table(df)
correlations

In [ ]:
worst_slices = slice_ranking(df, ["phase", "speed", "opening_name"], MIN_GROUP_SIZE)
worst_slices.head(20)

## Comparaison `Mistake` / `Blunder`

In [ ]:
quality_stats = group_quality_stats(df)
quality_stats

In [ ]:
plot_box_by_group(df, "cp_loss", "move_quality", PLOTS_DIR / "cp_loss_by_move_quality_box.png", "cp_loss par qualité de coup")

## Sauvegarde de tous les tableaux

In [ ]:
tables = {
    "numeric_summary": numeric_summary,
    "missingness": missingness,
    "flag_summary": flag_summary,
    "quality_stats": quality_stats,
    "phase_stats": phase_stats,
    "speed_stats": speed_stats,
    "result_stats": result_stats,
    "elo_bucket_stats": elo_bucket_stats,
    "transition_stats": transition_stats,
    "opening_name_stats": opening_name_stats,
    "top_hung_pieces": hung_piece_stats,
    "correlations": correlations,
    "worst_slices": worst_slices,
    "worst_examples": worst_examples,
    "king_table": king_table,
}
for name, table in tables.items():
    if isinstance(table, (pd.DataFrame, pd.Series)) and not table.empty:
        save_table(table, TABLES_DIR / f"{name}.csv")

for name, table in flag_tables.items():
    if isinstance(table, pd.DataFrame) and not table.empty:
        save_table(table, TABLES_DIR / f"{name}.csv")

sorted(p.name for p in TABLES_DIR.glob("*.csv"))[:20]